### Importing libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from services.api import *
from services.functions import *

# Dummy_json_data Analysis and Visualization

### Carts Dataframe Cleaning

In [ ]:
raw_carts_df=api_get_data(dataset="carts")

In [ ]:
carts_df = convert_to_pandas_df(raw_carts_df)

In [ ]:
len(carts_df)

In [ ]:
carts_df.rename(columns={
    "id": "cart_id",
    "userId": "user_id"
}, inplace=True)
carts_df

In [ ]:
cart_items_df = convert_to_pandas_df(
    raw_carts_df,
    record_path="products",
    meta=["id", "userId"],
    df_name="cart_"
)

In [ ]:
cart_items_df.rename(
    columns={
        "id": "product_id",
        "cart_userId": "user_id"
    },
    inplace=True
)
cart_items_df

In [ ]:
cart_items_df[cart_items_df.duplicated()]

In [ ]:
cart_items_df=remove_duplicates(cart_items_df)

In [ ]:
cart_items_df=cast_columns(cart_items_df,
                       columns_dict={
                           "cart_id":"int",
                           "user_id":"int"
                       }
                       )

In [ ]:
cart_items_df.head(10)

In [ ]:
cart_items_df,cart_inc_df=inconsistent_values(cart_items_df)

### Products Dataframe Cleansing

In [ ]:
raw_pr_df=api_get_data(dataset="products")

In [ ]:
products_df=convert_to_pandas_df(raw_pr_df,"products")

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
products_df.rename(
    columns={"id":"product_id"},
    inplace=True
)
products_df=products_df[
    ["product_id","title","category","price","discountPercentage","rating","stock","brand","shippingInformation","availabilityStatus","returnPolicy","minimumOrderQuantity"]
    ]


In [ ]:
products_df=remove_duplicates(products_df)

In [ ]:
products_df.info()

In [ ]:
products_df, inc_products_df=inconsistent_values(products_df)

### User Dataframe Cleansing

In [ ]:
raw_user=api_get_data(dataset="users")

In [ ]:
user_df=convert_to_pandas_df(raw_user,"users")
user_df.rename(
    columns={"id":"user_id"},
    inplace=True
)

In [ ]:
user_df=user_df[
    ["user_id",
     "firstName",
     "lastName",
     "age",
     "gender",
     "birthDate",
     "height",
     "weight",
     "eyeColor",
     "hair.color",
     "hair.type",
     "address.city",
     "address.stateCode",
     "bank.cardType",
     "bank.currency"]
]
user_df.head(10)

In [ ]:
user_df=remove_duplicates(user_df)

In [ ]:
user_df=cast_columns(user_df,columns_dict={"birthDate":"date"},date_format="ISO8601")
user_df,user_inc_df=inconsistent_values(user_df)

In [ ]:
user_df.info()

### Joining dataframes

In [ ]:
cart_products_merged=pd.merge(cart_items_df,products_df[["product_id","title","category","rating","brand","shippingInformation","returnPolicy"]], how="left")

In [ ]:
user_df.info()

In [ ]:
orders_merged=pd.merge(cart_products_merged,user_df[["user_id","age","gender","birthDate","height","weight","eyeColor","hair.color","hair.type","address.city","address.stateCode","bank.cardType","bank.currency"]],how="left")

In [ ]:
orders_merged.head(10)

### Dummy Json Data Analysis

Customers or products that did not purchased an order yet.

In [ ]:
orders_merged.info()

In [ ]:
inactive_cust_count = (
    (orders_merged["cart_id"].isnull()) &
    (orders_merged["user_id"].notnull())
).sum()

In [ ]:
if inactive_cust_count !=0:
    print(f"{inactive_cust_count} users have not purchased an order yet.")
else:
    print("All the signed users have proceed at least to one order.")


In [ ]:
not_purchased_products=(
    (orders_merged["cart_id"].isnull()) &
    (orders_merged["product_id"].notnull())
).sum()

In [ ]:
if not_purchased_products!=0:
    print(f"{not_purchased_products} products have not been yet placed in an order at all.")
else:
    print(f"All the products have been purchased at least once.")

Replacing the state codes with real full names.

In [ ]:
us_states = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming"
}

orders_merged["address.stateCode"] = (
    orders_merged["address.stateCode"]
    .str.strip()
    .str.upper()
    .replace(us_states)
)

In [ ]:
orders_merged.head(10)

Basic Report

In [ ]:
orders_basic_analysis=create_basic_analysis_table(orders_merged,"total","cart_id","category","bank.cardType","address.stateCode","State","card")
orders_basic_analysis.head(10)

In [ ]:
group_product=group_by(products_df,["returnPolicy"],{
    "rating":"mean",
    "price":"mean",
    "discountPercentage":"mean",
    "stock":"sum",
}
)
group_product.sort_values("rating")

In [ ]:
group_state=group_by(
    orders_merged,
    "address.stateCode",
    {
        "total":"sum",
        "cart_id":"count",
        "quantity":"sum",
        "discountPercentage":"mean",
        "discountedTotal":"sum",
        "age":"mean",
        } 
                     )
group_state.sort_values("discountedTotal",ascending=False,inplace=True)
group_state

In [ ]:
orders_merged.info()

In [ ]:
group_user=group_by(
    orders_merged,
    "user_id",
    {
    "discountedTotal":"sum",
    "cart_id":"count",
    "age":"mean"
    }
)
group_user.sort_values("discountedTotal",ascending=False)

### Dummy Json data visualization

In [ ]:
products_graph=create_bar_chart(
    dataframe=products_df,
    x_column="returnPolicy",
    y_column="stock",
    estimator="sum",
    chart_title="Average Stock by Return Policy",
    x_title="Return Policy",
    y_title="Total stock"
)

In [ ]:
state_graph=create_bar_chart(group_state.head(5),
                             "address.stateCode",
                             "discountedTotal","sum",
                             "Top 5 States by Revenue",
                             "State",
                             "Revenue",
                             )

In [ ]:
age_revenue_graph=create_scatterplot(
    group_user,
    "age",
    "discountedTotal",
    "Relatioship between age and Spending",
    "User's age",
    "Total Spending per user"
)

In [ ]:
histogram=create_histogram(orders_merged,"age")

In [ ]:
heatmap=create_heatmap(
    orders_merged,
    orders_merged.select_dtypes(include="number").columns
    )